# Adding Data to MIRA-DB: Guide and Explanations

This notebook demonstrates how to add data to the MIRA-DB database, including how to set up the database, insert extraction methods, query and update records, and process extracted model and metadata files. 

**How to use and modify:**
- Read the markdown cells for explanations of each code block.
- Adjust file paths, extraction methods, or PMIDs as needed for your data.
- Use the provided code as a template for your own data import or database update workflows.

---


## 1. Database Initialization

This cell sets up logging, imports the database manager, and initializes the database connection. It also creates the necessary tables if they do not exist and prepares the model manager for further operations.

**Modify as needed:**
- Change the database name in `get_db('primary')` if you want to use a different database.
- Adjust logging configuration as required for your environment.


In [ ]:
import logging
from miradb.db.manager import get_db, MiraModelManager

logger = logging.getLogger(__name__)

db = get_db('primary')

db.create_tables()

mira_db = MiraModelManager(db.host)

Table text_references already exists! No action taken.
Table extraction_method already exists! No action taken.
Table text_contents already exists! No action taken.
Table ode_expressions already exists! No action taken.
Table mira_template_models already exists! No action taken.


## 2. Inserting Extraction Methods

This cell demonstrates how to add new extraction methods to the database. Each method describes a way data can be extracted (e.g., text, image, marker, etc.).

**Modify as needed:**
- Add or remove `ExtractionMethod` entries to match your extraction pipelines.
- Update the descriptions to reflect your workflow.


In [ ]:
from sqlalchemy.orm import sessionmaker
from miradb.db.schema import ExtractionMethod

Session = sessionmaker(bind=db.engine)

with Session() as session:
    session.add(ExtractionMethod(extraction_method="marker", extraction_method_desc="Agentic Pipeline, Text, GPU, Marker"))
    session.add(ExtractionMethod(extraction_method="mineru_image", extraction_method_desc="Agentic Pipeline, Image, CPU, MinerU"))
    session.add(ExtractionMethod(extraction_method="mineru_text", extraction_method_desc="Agentic Pipeline, Text, CPU, MinerU"))
    session.add(ExtractionMethod(extraction_method="xml_extraction", extraction_method_desc="Agentic Pipeline, Text, CPU, pubmed xml file"))
    session.commit()

3


## 3. Querying Extraction Methods

This cell shows how to query the database for a specific extraction method and print its ID. This is useful for referencing extraction methods in other database operations.

**Modify as needed:**
- Change the method name in the query to look up a different extraction method.
- Use the result for linking to other tables or for debugging.


In [ ]:
from sqlalchemy.orm import sessionmaker
from miradb.db.schema import ExtractionMethod
from sqlalchemy import select

Session = sessionmaker(bind=db.engine)

with Session() as session:
    result = session.execute(select(ExtractionMethod.id).where(ExtractionMethod.extraction_method == "mineru_text")).scalar()
    print(result)
    session.commit()

## 4. (Optional) Dropping or Disposing Database

This cell contains commented-out code for disposing of the database engine or dropping tables. Use with caution—uncomment only if you need to reset or clean up your database.

**Modify as needed:**
- Uncomment lines to enable dropping tables or disposing the engine.
- Use only for development or maintenance, not in production.


In [ ]:
# db.engine.dispose()
# db.drop_tables()

True

## 5. Preparing Model Paths and Folders

This cell sets up paths to the extracted model data and lists available folders for processing. It uses `Path`, `pystow`, and `tqdm` for file management and progress tracking.

**Modify as needed:**
- Change `MODEL_PATH` to point to your data directory.
- Adjust folder filtering logic if your directory structure is different.


In [ ]:
from pathlib import Path
import json
from tqdm import tqdm


MODEL_PATH = Path.home() / "mira_data" / "paper_extraction"

folders = [f for f in MODEL_PATH.iterdir() if f.is_dir()]
folder_names = [f.name for f in folders]

## 6. Inserting Extracted Model Data into Database

This cell loops through a list of PMIDs, loads extracted model and intermediate data, and inserts them into the database. It handles different extraction methods and updates or creates references as needed.

**Modify as needed:**
- Update the `pmids` list to include your own PMIDs.
- Change the `method` variable to match your extraction method.
- Adjust file paths and error handling as required for your data structure.


In [ ]:
from sqlalchemy import select
from miradb.db.schema import ExtractionMethod

MODEL_PATH = Path.home() / ".data" / "mira" / "paper_extraction"

method = "xml_extraction"  # "marker", "mineru_image", or "mineru_text"

with Session() as session:
    extraction_id = session.execute(select(ExtractionMethod.id).where(ExtractionMethod.extraction_method == method)).scalar()

for pmid in tqdm(folder_names):

    try:
        model = MODEL_PATH / pmid / "tm" / method / f"{pmid}.json"
        with open(model, 'r') as f:
            data = json.load(f)

        data_path = MODEL_PATH / pmid / "tm" / method / f"{pmid}_intermediates.json"
        with open(data_path, 'r') as f:
            intermediates = json.load(f)

        text_ref = mira_db.get_text_ref(pmid=pmid)
        if text_ref is None:
            text_ref = mira_db.add_text_ref(pmid=pmid)
        else:
            text_ref = text_ref["id"]
        
        base = MODEL_PATH / str(pmid) 
        folder = next(p for p in base.iterdir() if p.is_dir() and p.name.startswith("P"))
        folder_path = str(folder.relative_to(base.parent))

        extracted_info_path = intermediates["ode"]["extraction_file"]
        
        if method == "marker":
            extracted_info_path = base.stem + "/" + method

        if "mineru" in method:
            if not folder:
                continue
            nxml_file = next((f for f in folder.iterdir() if f.suffix == ".nxml"), None)
            if not nxml_file:
                continue
            folder_name = nxml_file.stem
            extracted_info_path = base.stem + "/" + folder_name

        context_ref = mira_db.add_text_content(text_ref=text_ref, folder_path=folder_path, extraction_method_id=extraction_id, extracted_info_path=extracted_info_path)
        ode_id = mira_db.add_odes(txt_content_ref=int(context_ref), extraction_method_id=extraction_id, ode=intermediates["ode"]["ode_str"], corrected_ode=intermediates["ode"]["corrected_ode_str"])
        mira_db.add_tm(ode_ref=ode_id, grounded_concepts=intermediates["ode"]["concepts"], mira_template_model=data)

    except Exception as e:
        logger.error(f"Error processing PMID {pmid}: {e}")
        continue

## 7. Extracting and Updating Metadata from .nxml Files

This cell extracts metadata (title, authors, year, journal, DOI, PMC ID, keywords) from `.nxml` files for each PMID and updates the database accordingly.

**Modify as needed:**
- Ensure your `.nxml` files are organized as expected in the directory structure.
- Adjust the XML parsing logic if your files have a different schema.
- Add or remove metadata fields as needed for your application.


In [ ]:
# Script to extract metadata from .nxml files and update the database for each PMID in pmid_list. 

import os
import xml.etree.ElementTree as ET
import pystow

BASE = pystow.module("mira", "paper_extraction")

for pmid in folder_names:

    base_path = BASE.join(str(pmid))
    pmc_folder = next((p for p in base_path.iterdir() if p.is_dir() and p.name.startswith("P")), None)

    if not pmc_folder:
        print(f"No PMC folder found for PMID {pmid} in: {base_path}")
        continue
        
    dir_path = os.path.join(base_path, pmc_folder)

    nxml_file = next((f for f in os.listdir(dir_path) if f.endswith('.nxml')), None)

    if not nxml_file:
        print("No .nxml file found.")
    else:
        tree = ET.parse(os.path.join(dir_path, nxml_file))
        root = tree.getroot()

        def get_text(el):
            """Recursively get all text from an element."""
            return ' '.join(el.itertext()).strip() if el is not None else None

        # --- Title ---
        title_el = root.find('.//title-group/article-title')
        title = get_text(title_el)

        # --- Authors ---
        authors = []
        for contrib in root.findall('.//contrib[@contrib-type="author"]'):
            surname = get_text(contrib.find('name/surname'))
            given = get_text(contrib.find('name/given-names'))
            if surname and given:
                authors.append(f"{given} {surname}")
            elif surname:
                authors.append(surname)

        # --- Year of Publication ---
        pub_date = root.find('.//pub-date[@pub-type="epub"]') or \
                root.find('.//pub-date[@pub-type="ppub"]') or \
                root.find('.//pub-date')
        year = get_text(pub_date.find('year')) if pub_date is not None else None

        # --- Journal ---
        journal = get_text(root.find('.//journal-title'))

        # --- DOI ---
        doi = None
        for aid in root.findall('.//article-id'):
            if aid.attrib.get('pub-id-type') == 'doi':
                doi = aid.text
                break

        # --- PMC ID ---
        pmc_id = None
        for aid in root.findall('.//article-id'):
            if aid.attrib.get('pub-id-type') == 'pmc':
                pmc_id = aid.text
                break

        # --- Keywords ---
        keywords = [get_text(kw) for kw in root.findall('.//kwd')]

        mira_db.update_text_ref(pmid=pmid, title=title, authors=authors, year=year, journal=journal, doi=doi, pmcid=pmc_id, keywords=keywords)
